# 08 — Agents & inference-time compute: graphs that unroll at runtime

**What you'll learn**

- `dspy.ReAct`: the model decides *which tool to call and how many hops to take* — with the paper ↔ code map (Yao et al., 2022).
- Tools as plain Python functions, and what makes a function a *good* tool.
- Static vs. dynamic computation graphs — and why agents are still optimizable modules.
- Buying accuracy at inference time: `dspy.Refine` and `dspy.BestOfN` (the successors of the removed Assertions API).

*Time: ~25 min. Cost: ~$0.05. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, and the second line finds or starts the MLflow UI — click the printed link to browse them. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()
obs.mlflow_ui()            # prints the link where the traces appear

Boilerplate — the corpus and search function from notebook 03 (we won't need `HopQA` today; we're replacing its wiring). Skim and run.

In [ ]:
# boilerplate: corpus + search (same code as notebook 03, without HopQA)
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

## Un-hard-coding the graph

`HopQA` embodies a *decision we made*: exactly two hops, search after each. For Aldervane that was right. But some questions need one hop, some need three, some need a different tool entirely. You can keep growing `forward` with conditionals — or you can hand the control flow itself to the model.

That is **ReAct** (*ReAct: Synergizing Reasoning and Acting in Language Models*, Yao et al., 2022, [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)). The model alternates between thinking and acting, observing tool results, until it decides it can answer:

```
        +---> thought   ("I should find who keeps the lighthouse first")
        |        |
        |        v
        |     action    (call a tool:  search("Saltglass Lighthouse keeper"))
        |        |
        |        v
        |    observation (the tool's return value, fed back into context)
        +--------+
                 |   ...loop, until the model chooses to finish...
                 v
              answer
```

| ReAct paper concept | Plain English | Where it lives in code |
|---|---|---|
| Interleaved reasoning and acting traces | think → act → observe, repeated | the loop inside `dspy.ReAct`; visible in `pred.trajectory` |
| Action space | The discrete set of actions the model may take | `tools=[...]` — plain functions, auto-wrapped into `dspy.Tool` |
| Observation | Environment feedback appended to context | each tool's return value (`observation_i` in the trajectory) |
| Episode termination | The agent decides it is done | the built-in finish action; hard cap via `max_iters` |

## Tools are just functions (with manners)

A tool is a Python callable. Its **name**, **docstring**, and **type hints** are not decoration — they are compiled into the prompt that tells the model what the tool does and how to call it. Writing a good tool is writing a good signature, in function form:

In [ ]:
def search_archipelago(query: str) -> str:
    """Search the Aldervane corpus for passages matching the query. Returns the two best matches."""
    return "\n".join(f"{p['title']}: {p['text']}" for p in search(query, k=2))

agent = dspy.ReAct("question -> answer", tools=[search_archipelago], max_iters=5)

pred = agent(question="What instrument was invented by the current keeper of the Saltglass Lighthouse?")
print(pred.answer, "\n")

for step, value in pred.trajectory.items():
    print(f"{step:16s} {str(value)[:100]}")

> **What you should see:** the right answer (tidewriter) and — the real payload — the trajectory: `thought_0` planning the first lookup, `tool_name_0` / `tool_args_0` calling your function, `observation_0` with the passages, then a second cycle targeting *Mara Venn*, then finish.
>
> Compare with notebook 05: there, the bridge-entity second hop had to be *learned into* `hop2`'s demos. Here the model *improvises* it at runtime — no demos, but two extra LM round-trips and no guarantee it happens every time. You are looking at the same capability bought with different currency.

## Static vs. dynamic graphs

`HopQA`'s graph is fixed: three LM nodes, two search nodes, every run (the PyTorch analogy: a fixed architecture). ReAct's graph **unrolls per input** — one question might use one tool call, another four (the analogy: an RNN unrolled to variable length, where the *model* picks the length).

The trade, stated plainly:

| | static (`HopQA`) | dynamic (`ReAct`) |
|---|---|---|
| cost | fixed, predictable | variable, higher on average |
| debuggability | every node known in advance | read the trajectory after the fact |
| flexibility | handles what you anticipated | routes around surprises |
| optimizability | per-node θ, cleanly addressable | still optimizable — see below |

That last row matters: `ReAct` is a `dspy.Module` like any other. It has `named_predictors()`, so **everything from notebooks 05–07 applies** — bootstrap successful trajectories into demos, or point GEPA's reflection at its instructions. Agents are not outside the optimization story; they are just modules with loops in `forward`.

**Cousin, briefly — `dspy.ProgramOfThought`** (*Program of Thoughts Prompting*, Chen et al., 2022, [arXiv:2211.12588](https://arxiv.org/abs/2211.12588)): instead of reasoning in prose, the model writes *Python*, and an interpreter runs it — "reasoning as executable code, computation delegated to an interpreter" ↔ `dspy.ProgramOfThought(signature)`. Ideal when the hard part is arithmetic. Caveat before you run it: in DSPy 3.x the generated code executes in a **Deno-sandboxed interpreter**, so it needs [Deno](https://deno.land) installed:

```python
pot = dspy.ProgramOfThought("question -> answer: float")
pot(question="A lighthouse lens rotates 4.5 times/min. How many rotations in 2h20m?")
```

## Buying accuracy at inference time

Optimizers spend compute **once, at compile time**, to make every future call better. There is a complementary knob: spend more compute **per call**. Sample several attempts, keep the best — no training, no artifacts, just a better answer *now*, at N× the price.

(History box: DSPy 2.x had `dspy.Assert`/`dspy.Suggest` for "retry until the output satisfies a check". They were **removed in 3.x**; what follows is the modern replacement.)

Both replacements share a shape: a module to wrap, a reward function `reward_fn(args: dict, pred) -> float` (note: it receives the call's *input kwargs*, not an Example — there is no dataset at inference time!), a `threshold` to hit, and up to `N` attempts at varied sampling:

- **`dspy.BestOfN`** — run up to N times, return the first attempt clearing `threshold` (or the best seen).
- **`dspy.Refine`** — same, but between attempts an LM reads the previous attempt + its reward and writes *advice* fed into the next attempt. Sampling with a memory.

In [ ]:
def concise_reward(args, pred):
    """Reward answers of four words or fewer (our exact-match style rule, as a check)."""
    return float(len(pred.answer.split()) <= 4)

refined = dspy.Refine(
    module=dspy.ChainOfThought("context: list[str], question: str -> answer"),
    N=3,
    reward_fn=concise_reward,
    threshold=1.0,
)

context = as_context(search("Mara Venn tidewriter", k=2))
refined(context=context, question="What did Mara Venn invent, and where does she sell it?")

> **What you should see:** a terse answer. The bait in the question ("...and where does she sell it?") pushes the model toward a long reply; if attempt 1 is verbose, the reward is 0.0, and `Refine` retries with generated advice about brevity until an attempt clears the threshold (or N runs out — by default it then returns the best attempt).
>
> When to reach for which knob: **compile-time optimization** when you have data and traffic to amortize over; **inference-time compute** when calls are rare, stakes are high, or you need a guardrail expressed as code (a validity check as `reward_fn`). They stack — an optimized module wrapped in `Refine` is a perfectly good design.

**Roll-call** — other composition tools that ship with DSPy, one line each, so you know they exist: `dspy.CodeAct` (ReAct where actions are code), `dspy.Parallel` (fan out sub-calls), `dspy.MultiChainComparison` (generate k reasoning chains, have a judge synthesize), `dspy.majority` (self-consistency voting), `dspy.streamify` / `dspy.asyncify` (streaming and async wrappers for serving).

## Recap

| Concept | One-liner |
|---|---|
| `dspy.ReAct(sig, tools=[...])` | model-controlled think → act → observe loop (Yao et al. 2022) |
| tools | plain functions; name + docstring + type hints become the tool spec |
| `pred.trajectory` | the unrolled graph, readable after the fact |
| static vs dynamic | fixed cost & clean θ vs. runtime flexibility — same optimizers apply to both |
| `dspy.ProgramOfThought` | reasoning as executable code (Chen et al. 2022); needs Deno in 3.x |
| `dspy.Refine` / `dspy.BestOfN` | inference-time retries against `reward_fn(args, pred)`; Assertions' successors |

## Exercises

1. Add a second tool `lookup_by_title(title: str) -> str` (exact title match in `CORPUS`) and ask a question whose natural solution uses both tools. Read the trajectory: did the agent choose sensibly?
2. Set `max_iters=2` and ask a 2-hop question. Watch the graceful failure — then explain why notebook 03's *static* pipeline cannot fail in this particular way (and what it pays for that safety).
3. Write a `reward_fn` that rejects answers beginning with "The " and wrap `BestOfN` around the agent. You have layered a *runtime* style rule on top of a *dynamic* graph — no optimizer involved.

**Next up:** the tour ends with the map — every optimizer family (including the weights track we haven't touched), what each one tunes, and a decision guide for real projects.